# 🧬 Enformer Tools Example

This notebook demonstrates how to run **Enformer** for regulatory activity prediction and variant-effect style analysis.

## 📖 What is Enformer?

[Enformer](https://www.nature.com/articles/s41592-021-01252-x) is a long-context DNA model for predicting regulatory activity from sequence.

### Key Features:
- **Regulatory prediction** -- Predicts binned regulatory signal from 196,608 bp input windows
- **Species-aware outputs** -- Supports `human` and `mouse` output heads
- **Track selection** -- Extract only tracks needed for downstream analysis
- **GPU-ready** -- Works with local venv execution

## 📥 Imports

In [1]:
import numpy as np

from bio_programming_tools import (
    ENFORMER_CONTEXT,
    EnformerConfig,
    EnformerInput,
    run_enformer,
)


## 🧪 1. Predict Regulatory Activity

Run Enformer on a single 196,608 bp sequence and extract selected tracks.

### API Reference

**`EnformerInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequence` | `str` | DNA sequence to score. Must be exactly 196,608 bp and contain only valid nucleotide characters (A/C/G/T/N). |

**`EnformerConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `output_tracks` | `List[int]` | *(required)* | Track indices to extract from model output |
| `species` | `Literal["human", "mouse"]` | `"human"` | Species track head to use |

**`EnformerOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequence` | `str` | Input DNA sequence that was scored |
| `sequence_length` | `int` | Length of the input sequence (always 196,608) |
| `prediction` | `List[List[float]]` | Predicted activity matrix with shape [896, num_tracks] |
| `output_tracks` | `List[int]` | Track indices extracted from Enformer |
| `species` | `str` | Species used for prediction (`"human"` or `"mouse"`) |

In [2]:
sequence = "ATCG" * (ENFORMER_CONTEXT // 4)

inputs = EnformerInput(sequence=sequence)
config = EnformerConfig(
    output_tracks=[0, 1, 2],
    species="human",
    verbose=False,
)

pred_result = run_enformer(inputs, config)
print(f"Output bins: {len(pred_result.prediction)}")
print(f"Tracks: {len(pred_result.prediction[0])}")


Output bins: 896
Tracks: 3


## 🧬 2. Variant-Effect Style Comparison

Run reference and alternate windows and compare predictions using log2 fold-change.


In [3]:
center = ENFORMER_CONTEXT // 2
ref_sequence = sequence
alt_base = "A" if ref_sequence[center] != "A" else "C"
alt_sequence = ref_sequence[:center] + alt_base + ref_sequence[center + 1:]

ref_result = run_enformer(EnformerInput(sequence=ref_sequence), config)
alt_result = run_enformer(EnformerInput(sequence=alt_sequence), config)

ref_pred = np.array(ref_result.prediction)
alt_pred = np.array(alt_result.prediction)
log2_fc = np.log2((alt_pred + 1e-6) / (ref_pred + 1e-6))
print(f"Max |log2 FC|: {np.abs(log2_fc).max():.4f}")


Max |log2 FC|: 0.0962


## 💾 3. Export Results

Save prediction outputs to the local example output directory.

### Supported formats:
- **JSON**
- **CSV**

In [4]:
pred_result.export("example_output/enformer_prediction", file_format="json")
print("Prediction exported to example_output/enformer_prediction.json")

ref_result.export("example_output/enformer_ref", file_format="csv")
print("Reference summary exported to example_output/enformer_ref.csv")


Prediction exported to example_output/enformer_prediction.json
Reference summary exported to example_output/enformer_ref.csv
